In [141]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd


**Set Up**

In [142]:
fn = 'resources/DK_test/networks/base_s_2__12h_2050.nc'


In [143]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.denmark.yaml").read_text())


INFO:pypsa.network.io:New version 1.0.5 available! (Current: 0.35.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, links, loads, stores


In [144]:
p = Path(fn)  
try:
    if p.exists():
        p.unlink()
        print(f"Deleted {p}")
    else:
        print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/DK_test/networks/base_s_2__12h_2050.nc


In [ ]:
n.remove(
                            "Generator",
                            name="DK1 0 4 onwind",
                    )

**Options**

In [ ]:
ongrid=True
cluster_cost_reduction=0.20
cluster_size=1229.64   #MW, it's the maximum installable capacity for each renewable in renewables in each country cluster
renewables={"solar", "onwind"}

In [146]:
n.buses


,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
EU,1.0,,-5.500000,46.000000,none,,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
co2 atmosphere,1.0,,-5.500000,46.000000,co2,t_co2,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
DK0 0 co2 stored,1.0,,9.648393,55.893047,co2 stored,t_co2,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 co2 stored,1.0,,12.303316,55.515974,co2 stored,t_co2,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 co2 sequestered,1.0,,9.648393,55.893047,co2 sequestered,t_co2,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 co2 sequestered,1.0,,12.303316,55.515974,co2 sequestered,t_co2,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 gas,1.0,,9.648393,55.893047,gas,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN


**Buses and Generators of the Cluster Addition**

In [147]:


def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    countries_renewables_cf = {}                #dictionary of dataframes by country and renewable type, sorting the generators by average capacity factor (ascending order)
    clusters_generators={}                      #dictionary of dataframes by country and renewable type, containing the generators assigned to the cluster  

    for country in config['countries']:
        for renewable in renewables:

            countries_renewables_cf[(country, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{country}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )

            clusters_generators[(country, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            countries_renewables_cf[(country, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{country}.*{renewable}$")].mean()
            countries_renewables_cf[(country, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{country}.*{renewable}$")]

            countries_renewables_cf[(country, renewable)] = countries_renewables_cf[(country, renewable)].sort_values("p_max_pu", ascending=False)

            #print(countries_renewables_cf[(country, renewable)])

            number_gen=0



            while  countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen >= len(countries_renewables_cf[(country, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(country, renewable)]  = n.generators.loc[countries_renewables_cf[(country, renewable)].index[0:number_gen+1]]
            remaining_capacity = countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #countries_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(country, renewable)].loc[clusters_generators[(country, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(country, renewable)])

            for idx in clusters_generators[(country, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " cluster"}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(country, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(country, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(country, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(country, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(country, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(country, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(country, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(country, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(country, renewable)].loc[idx].efficiency,
                    p_nom_extendable=True,
                    overwrite=True,)
                
                n.generators_t['p_max_pu'][clusters_generators[(country, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(country, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2 cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " H2"}", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery cluster"}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(country, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(country, renewable)].loc[idx].bus + " battery"}", "substation_off"],
                    )



                if idx == countries_renewables_cf[(country, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = countries_renewables_cf[(country, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(country, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(country, renewable)].loc[idx].name,
                    )

    nodes_with_clusters = (n.buses.loc[n.buses.index.str.contains("cluster"), [ "country", "location"]].drop_duplicates().reset_index(drop=True)
)



            

    return n, nodes_with_clusters

n, nodes_with_clusters = assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




Remaining top solar capacity outside the cluster: 637.6152340799661 MW
Capacity of the last solar generator adjusted to fit cluster size: 334.0689504498505 MW
                 bus control type  p_nom  p_nom_mod  p_nom_extendable  \
Generator                                                               
DK0 0 4 solar  DK0 0      PQ         7.0        0.0              True   
DK1 0 4 solar  DK1 0      PQ         0.0        0.0              True   

               p_nom_min  p_nom_max  p_min_pu  p_max_pu  ...  up_time_before  \
Generator                                                ...                   
DK0 0 4 solar        7.0  165.93105       0.0       1.0  ...               1   
DK1 0 4 solar        0.0  334.06895       0.0       1.0  ...               1   

               down_time_before  ramp_limit_up  ramp_limit_down  \
Generator                                                         
DK0 0 4 solar                 0            NaN              NaN   
DK1 0 4 solar             

In [148]:
n.buses

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
EU,1.0,,-5.500000,46.000000,none,,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
co2 atmosphere,1.0,,-5.500000,46.000000,co2,t_co2,EU,1.0,0.0,inf,PQ,,,,NaN,NaN
DK0 0 co2 stored,1.0,,9.648393,55.893047,co2 stored,t_co2,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK0 0 H2 cluster,1.0,,9.648393,55.893047,H2,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 battery cluster,1.0,,9.648393,55.893047,battery,MWh_el,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 cluster,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0


**Links of the Cluster Addition**

In [149]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
DK0 0 co2 sequestered,DK0 0 co2 stored,DK0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 co2 sequestered,DK1 0 co2 stored,DK1 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK0 0 OCGT,DK0 0 gas,DK0 0,,OCGT,0.410000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 OCGT,DK1 0 gas,DK1 0,,OCGT,0.410000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK1 0 urban decentral biomass boiler,DK1 0 solid biomass,DK1 0 urban decentral heat,,urban decentral biomass boiler,0.860000,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 urban decentral gas boiler,DK1 0 gas,DK1 0 urban decentral heat,,urban decentral gas boiler,0.980000,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 urban decentral resistive heater,DK1 0 low voltage,DK1 0 urban decentral heat,,urban decentral resistive heater,0.900000,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000


In [150]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters["location"]:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        link_name = f"{node} methanolisation"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"],
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )


    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

**Storages of the Cluster Addition**

In [151]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters["location"]:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





In [152]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [153]:
n.links.loc[n.links["bus1"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 solid biomass biomass-to-methanol,DK0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 solid biomass biomass-to-methanol,DK1 0 solid biomass,EU methanol,,biomass-to-methanol,0.6100,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation,DK0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 methanolisation,DK1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 methanolisation cluster,DK0 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 methanolisation cluster,DK1 0 H2 cluster,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [154]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [155]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Electrolysis,DK0 0,DK0 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 H2 Electrolysis,DK1 0,DK1 0 H2,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK0 0 H2 Electrolysis cluster,DK0 0 cluster,DK0 0 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK1 0 H2 Electrolysis cluster,DK1 0 cluster,DK1 0 H2 cluster,,H2 Electrolysis,0.6217,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [156]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
Bus,,,,,,,,,,,,,,,,
DK0 0 cluster,380.0,,9.648393,55.893047,AC,MWh_el,DK0 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK0 0 H2 cluster,1.0,,9.648393,55.893047,H2,MWh_LHV,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK0 0 battery cluster,1.0,,9.648393,55.893047,battery,MWh_el,DK0 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 cluster,380.0,,12.303316,55.515974,AC,MWh_el,DK1 0,1.0,0.0,inf,Slack,,,DK,1.0,1.0
DK1 0 H2 cluster,1.0,,12.303316,55.515974,H2,MWh_LHV,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN
DK1 0 battery cluster,1.0,,12.303316,55.515974,battery,MWh_el,DK1 0,1.0,0.0,inf,PQ,,,DK,NaN,NaN


In [157]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
DK0 0 H2 Store cluster,DK0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,119.238216,0.0,True,0,inf,0.0,NaN
DK0 0 battery cluster,DK0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,10366.102696,0.0,True,0,inf,0.0,NaN
DK1 0 H2 Store cluster,DK1 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,119.238216,0.0,True,0,inf,0.0,NaN
DK1 0 battery cluster,DK1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,10366.102696,0.0,True,0,inf,0.0,NaN


In [158]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
relation/5487095-400-DC-reversed,DK1 0,DK0 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,True,171.53249


In [159]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_min_pu,e_max_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
Store,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.000000,0.0,True,0.0,inf,-1.0,1.0,...,0.0,0.0,0.0,0.000000,0.000000,True,0,inf,0.0,
DK0 0 co2 stored,DK0 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK1 0 co2 stored,DK1 0 co2 stored,,co2 stored,0.000000,0.0,True,0.0,inf,0.0,1.0,...,0.0,0.0,0.0,247.607546,0.000000,True,0,inf,0.0,
DK0 0 co2 sequestered,DK0 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,7.374268e+08,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK1 0 co2 sequestered,DK1 0 co2 sequestered,,co2 sequestered,0.000000,0.0,True,0.0,8.677499e+07,0.0,1.0,...,-0.1,0.0,0.0,30.000000,0.000000,True,0,50.0,0.0,
DK0 0 gas Store,DK0 0 gas,,gas,0.000000,0.0,True,3805600.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK1 0 gas Store,DK1 0 gas,,gas,0.000000,0.0,True,6334336.0,inf,0.0,1.0,...,0.0,0.0,0.0,17.851178,0.000000,True,0,inf,0.0,
DK0 0 H2 Store,DK0 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.013398e+08,0.0,1.0,...,0.0,0.0,0.0,149.047770,0.000000,True,0,100.0,0.0,
DK1 0 H2 Store,DK1 0 H2,,H2 Store,0.000000,0.0,True,0.0,2.879317e+06,0.0,1.0,...,0.0,0.0,0.0,149.047770,0.000000,True,0,100.0,0.0,


In [160]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
DK0 0 OCGT methanol,EU methanol,DK0 0,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
DK1 0 OCGT methanol,EU methanol,DK1 0,,OCGT methanol,0.41,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.0


In [161]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
Carrier,,,,,
AC,0.0,#70af1d,AC,inf,0.0
DC,0.0,#8a1caf,DC,inf,0.0
offwind-ac,0.0,#6895dd,Offshore Wind (AC),inf,0.0
onwind,0.0,#235ebc,Onshore Wind,inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
...,...,...,...,...,...
agriculture electricity,0.0,#494778,agriculture electricity,inf,0.0
solar rooftop,0.0,#ffea80,solar rooftop,inf,0.0
urban central heat vent,0.0,#a74747,urban central heat vent,inf,0.0


In [162]:
n.global_constraints

,type,investment_period,carrier_attribute,sense,constant,mu
GlobalConstraint,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,"AC, DC",<=,1.029195e+05,0.0
biomass limit,operational_limit,NaN,solid biomass,<=,1.219759e+07,0.0
CO2Limit,co2_atmosphere,NaN,co2_emissions,<=,0.000000e+00,0.0


In [163]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
Load,,,,,,,
DK0 0,DK0 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK1 0,DK1 0 low voltage,electricity,,0.000000,0.0,-1.0,True
DK0 0 land transport EV,DK0 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK1 0 land transport EV,DK1 0 EV battery,land transport EV,,0.000000,0.0,-1.0,True
DK0 0 urban central heat,DK0 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK1 0 urban central heat,DK1 0 urban central heat,urban central heat,,0.000000,0.0,-1.0,True
DK0 0 solid biomass for industry,DK0 0 solid biomass for industry,solid biomass for industry,,541.095890,0.0,-1.0,True
DK1 0 solid biomass for industry,DK1 0 solid biomass for industry,solid biomass for industry,,245.433790,0.0,-1.0,True
DK0 0 gas for industry,DK0 0 gas for industry,gas for industry,,155.251142,0.0,-1.0,True


In [164]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,underground,under_construction,tags,geometry,dc,underwater_fraction,energy to power ratio,location,reversed,length_original
Link,,,,,,,,,,,,,,,,,,,,,
relation/5487095-400-DC,DK0 0,DK1 0,,DC,0.976096,True,0,inf,600.0,0.0,...,1.0,0.0,relation/5487095,LINESTRING (10.505724427906852 55.365970143543...,1.0,0.560249,NaN,,False,171.53249
DK0 0 co2 sequestered,DK0 0 co2 stored,DK0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 co2 sequestered,DK1 0 co2 stored,DK1 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK0 0 OCGT,DK0 0 gas,DK0 0,,OCGT,0.410000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
DK1 0 OCGT,DK1 0 gas,DK1 0,,OCGT,0.410000,True,0,25.0,0.0,0.0,...,NaN,NaN,,,NaN,NaN,NaN,,False,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DK1 0 electricity cluster back,DK1 0,DK1 0 cluster,,AC,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
DK0 0 battery charger cluster,DK0 0 cluster,DK0 0 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
DK0 0 battery discharger cluster,DK0 0 battery cluster,DK0 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


**Exporting**

In [165]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network'saved to 'resources/DK_test/networks/base_s_2__12h_2050.nc contains: generators, links, global_constraints, carriers, stores, loads, buses


<xarray.Dataset> Size: 517kB
Dimensions:                               (snapshots: 730,
                                           investment_periods: 0,
                                           generators_i: 61,
                                           generators_t_p_max_pu_i: 46,
                                           links_i: 145,
                                           links_t_efficiency_i: 8,
                                           links_t_p_max_pu_i: 4,
                                           global_constraints_i: 3,
                                           carriers_i: 118, stores_i: 32,
                                           stores_t_e_min_pu_i: 2,
                                           stores_t_e_max_pu_i: 4, loads_i: 37,
                                           loads_t_p_set_i: 10, buses_i: 64)
Coordinates: (12/15)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * investment_periods                    (investment_periods) object 0B 
  * generators_i                          (generators_i) object 488B 'DK0 0 0...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 368B ...
  * links_i                               (links_i) object 1kB 'relation/5487...
  * links_t_efficiency_i                  (links_t_efficiency_i) object 64B '...
    ...                                    ...
  * stores_i                              (stores_i) object 256B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 16B 'D...
  * stores_t_e_max_pu_i                   (stores_t_e_max_pu_i) object 32B 'D...
  * loads_i                               (loads_i) object 296B 'DK0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 80B 'DK0 0...
  * buses_i                               (buses_i) object 512B 'DK0 0' ... '...
Data variables: (12/90)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    investment_periods_objective          (investment_periods) float64 0B 
    investment_periods_years              (investment_periods) float64 0B 
    ...                                    ...
    buses_unit                            (buses_i) object 512B 'MWh_el' ... ...
    buses_location                        (buses_i) object 512B 'DK0 0' ... '...
    buses_control                         (buses_i) object 512B 'Slack' ... 'PQ'
    buses_country                         (buses_i) object 512B 'DK' ... 'DK'
    buses_substation_lv                   (buses_i) float64 512B 1.0 1.0 ... nan
    buses_substation_off                  (buses_i) float64 512B 1.0 1.0 ... nan
Attributes:
    network__multi_invest:  0
    network_name:           Unnamed Network
    network_pypsa_version:  0.35.2
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...